<a href="https://colab.research.google.com/github/miriamamin1213-ux/Seed42_Models/blob/main/Generric_Transformer_Hecktor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, balanced_accuracy_score,f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" # Required for newer PyTorch versions
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    np.random.seed(seed)
    random.seed(seed)


SEED = 42
seed_everything(seed = SEED)

import pandas as pd

sheet_id = "1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv"

url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)

print(df.shape)
df.head()


df = df.dropna(subset=['HPV Status'])
tobacco_mode = df['Tobacco Consumption'].mode()[0]

df['Tobacco Consumption'] = df[
    'Tobacco Consumption'
].fillna(tobacco_mode)

alcohol_mode = df['Alcohol Consumption'].mode()[0]

df['Alcohol Consumption'] = df[
    'Alcohol Consumption'
].fillna(alcohol_mode)

df = df.drop(
    columns=[
        'PatientID',
        'CenterID',
        'Task 1',
        'Task 2',
        'Task 3'
    ]
)

df = df.dropna()
print(df.shape)

df['T-stage'] = df['T-stage'].replace({
    'T0':0,
    'T1':1,
    'T2':2,
    'T3':3,
    'T4':4
})

df['N-stage'] = df['N-stage'].replace({
    'N0':0,
    'N1':1,
    'N2':2,
    'N3':3
})

df['M-stage'] = df['M-stage'].replace({'M0':0,'M1':1})

X = df[
    [
        'Age',
        'Gender',
        'Tobacco Consumption',
        'Alcohol Consumption',
        'Performance Status',
        'Relapse',
        'RFS',
        'Treatment',
        'T-stage',
        'N-stage',
        'M-stage'
    ]
]

y = df['HPV Status']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED
)

print('train sample:',y_train.value_counts(),'\n')
print('test sample:', y_test.value_counts())

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
smote = SMOTE(random_state=SEED)
X_train_smote, y_train_smote = smote.fit_resample( X_train,y_train)
print('counting sample:', pd.Series(y_train_smote).value_counts())

#fixing data imbalance with smote
X_train_smote = torch.FloatTensor(X_train_smote)
y_train_smote = torch.LongTensor(y_train_smote.to_numpy())
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor( y_test.to_numpy())

#Model Architecture
# Transformer-based model for tabular classification
class HPVNet_Transformer(nn.Module):

    def __init__(
        self,
        num_features=11,
        d_model=32,
        nhead=4,
        num_layers=2,
        dim_feedforward=64,
        dropout=0.1,
        num_classes=2
    ):
        super().__init__()

        self.num_features = num_features
        self.d_model = d_model

        # Convert each scalar feature into a d_model-dimensional token
        self.feature_projection = nn.Linear(1, d_model)

        # Learnable embedding identifying each feature
        self.feature_embedding = nn.Parameter(
            torch.zeros(1, num_features, d_model)
        )

        # Learnable classification token
        self.cls_token = nn.Parameter(
            torch.zeros(1, 1, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.feature_embedding, mean=0.0, std=0.02)
        nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

        nn.init.xavier_uniform_(self.feature_projection.weight)
        nn.init.zeros_(self.feature_projection.bias)

    def forward(self, x):
        batch_size = x.size(0)

        # Input: [batch_size, 11]
        # Tokens: [batch_size, 11, d_model]
        x = x.unsqueeze(-1)
        x = self.feature_projection(x)

        # Distinguish Age token from Gender, Tobacco, T-stage, etc.
        x = x + self.feature_embedding

        # Add classification token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Model interactions between tabular features
        x = self.transformer(x)

        # Use the CLS token for classification
        x = self.norm(x[:, 0])

        logits = self.classifier(x)

        return logits

#Defining model, and optimisation function/parameters
model = HPVNet_Transformer()
weights = torch.tensor([3.0,1.0], dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
# optimizer = torch.optim.AdamW(model.parameters(),lr=0.001, weight_decay=0.01)

#Training the model
epochs = 500
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(epochs):
    model.train()
    outputs = model(X_train_smote)
    train_loss = criterion(outputs, y_train_smote)
    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()
    model.eval()

    with torch.no_grad():
        test_outputs = model(X_test)
        test_loss = criterion(test_outputs, y_test)

    if test_loss.item() < best_val_loss:

        best_val_loss = test_loss.item()
        best_epoch = epoch + 1
        torch.save(model.state_dict(), "best_model_transformer.pth")

    if (epoch+1) % 50 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train={train_loss.item():.4f}, "
            f"Test={test_loss.item():.4f}",
            f"Best Epoch={best_epoch}"
        )

print("Best Test Loss =", best_val_loss)
print("Best Epoch =", best_epoch)


#Inferencing the mdoel
model.load_state_dict(torch.load("best_model_transformer.pth"))
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    probabilities = torch.softmax(outputs, dim=1)  # Convert logits to probabilities
    predicted = torch.argmax(outputs,dim=1)

results = classification_report(y_test.numpy(),predicted.numpy(),digits=4)
print('results:',results)

# Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test.numpy(),predicted.numpy())
f1 = f1_score(y_test.numpy(), predicted.numpy())

# auc = roc_auc_score(y_test.numpy(), probabilities.numpy())
y_prob = probabilities.cpu().numpy()
if y_prob.shape[1] == 2:   # Binary classification
    auc = roc_auc_score(y_test.numpy(), y_prob[:, 1])
else:                      # Multi-class classification
    auc = roc_auc_score(
        y_test.numpy(),
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1-score:          {f1:.4f}")
print(f"AUC:               {auc:.4f}")

(726, 17)
(423, 12)
train sample: HPV Status
1.0    320
0.0     18
Name: count, dtype: int64 

test sample: HPV Status
1.0    80
0.0     5
Name: count, dtype: int64
counting sample: HPV Status
1.0    320
0.0    320
Name: count, dtype: int64


/tmp/ipykernel_873/2017862917.py:71: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['T-stage'] = df['T-stage'].replace({
/tmp/ipykernel_873/2017862917.py:79: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['N-stage'] = df['N-stage'].replace({
/tmp/ipykernel_873/2017862917.py:86: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silen

Epoch 50, Train=0.3194, Test=0.7175 Best Epoch=1
Epoch 100, Train=0.1704, Test=0.5693 Best Epoch=94
Epoch 150, Train=0.1214, Test=0.4983 Best Epoch=150
Epoch 200, Train=0.0924, Test=0.4917 Best Epoch=155
Epoch 250, Train=0.0802, Test=0.5303 Best Epoch=236
Epoch 300, Train=0.0839, Test=0.4722 Best Epoch=236
Epoch 350, Train=0.0547, Test=0.5151 Best Epoch=236
Epoch 400, Train=0.0473, Test=0.5589 Best Epoch=236
Epoch 450, Train=0.0588, Test=0.5909 Best Epoch=236
Epoch 500, Train=0.0333, Test=0.6517 Best Epoch=236
Best Test Loss = 0.44968029856681824
Best Epoch = 236
results:               precision    recall  f1-score   support

           0     0.3333    0.6000    0.4286         5
           1     0.9737    0.9250    0.9487        80

    accuracy                         0.9059        85
   macro avg     0.6535    0.7625    0.6886        85
weighted avg     0.9360    0.9059    0.9181        85

Balanced Accuracy: 0.7625
F1-score:          0.9487
AUC:               0.8575
